# Subastas Casanare

Version: 1.6.0

Datos en Google Drive: **`MyDrive/subasta_ganadera_data/`**

## Uso en Colab

1. **Celda A** — monta Drive, funciones y precarga.
2. **Celda B** — panel (Descarga · Datos · Exportar).
3. **Celda C** — graficos (celda aparte; funciona en Colab).


In [ ]:
# Celda A - Configuracion (montar Drive + funciones)
!pip install -q pandas pdfplumber requests pyarrow plotly ipywidgets kaleido

import re
from pathlib import Path

import pandas as pd
import pdfplumber
import requests
from google.colab import drive

drive.mount("/content/drive")

PROYECTO = Path("/content/drive/MyDrive/subasta_ganadera_data")
LOTE_DIR = PROYECTO / "data" / "lotes"
RESUMEN_DIR = PROYECTO / "data" / "resumen"
COMBINED_DIR = PROYECTO / "data" / "combined"
EXPORT_DIR = PROYECTO / "export"
PDF_DIR = PROYECTO / "pdfs"
for d in [LOTE_DIR, RESUMEN_DIR, COMBINED_DIR, EXPORT_DIR, PDF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ROW_PATTERN = re.compile(
    r"^(\d+)\s+([A-Z]{2})\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"(.+?)\s+(\d{2}:\d{2}:\d{2})\s+([\d.]+)\s+([\d.]+)(?:\s+(.*))?$"
)
SUMMARY_ROW_PATTERN = re.compile(
    r"^(.+?)\s+([A-Z]{2})\s+(\d+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)$"
)
HEADER_PATTERN = re.compile(
    r"FERIA NO\.\s*(\d+).*?CIUDAD\s+(.+?)\s+FECHA FERIA\.\s*(\d{4}-\d{2}-\d{2})", re.DOTALL
)
SECTION_PATTERN = re.compile(r"^(REMATE|SUBASTA)\s+(.+)$")
LOTES_SKIP = ("SUBASTA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS", "TOTALES:", "REMATE")
RESUMEN_SKIP = ("SUBASTA GANADERA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS GENERALES", "Edad Sexo", "TOTALES:")


def parse_num(v):
    return int(str(v).replace(".", ""))


def listar_ferias_en_drive():
    return sorted(int(p.stem.replace("feria_", "")) for p in LOTE_DIR.glob("feria_*.parquet"))


def ultima_feria_en_drive():
    ferias = listar_ferias_en_drive()
    return ferias[-1] if ferias else 0


def download_pdf(feria: int) -> Path:
    url = f"https://www.subacasanare.com/Precio_Pdf/{feria}"
    dest = PDF_DIR / f"{feria}.pdf"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("respuesta no es PDF")
    dest.write_bytes(r.content)
    return dest


def extract_metadata(text: str) -> dict:
    m = HEADER_PATTERN.search(text)
    if not m:
        return {}
    return {"feria_no": m.group(1), "ciudad": m.group(2).strip(), "fecha_subasta": m.group(3)}


def extract_lotes(pdf_path: Path) -> pd.DataFrame:
    records, current, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                if not line or in_summary or line.startswith(LOTES_SKIP):
                    continue
                match = ROW_PATTERN.match(line)
                if match:
                    if current:
                        records.append(current)
                    current = {
                        "pagina": page_num, "lote": int(match.group(1)), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_total": parse_num(match.group(4)),
                        "peso_promedio": parse_num(match.group(5)), "procedencia": match.group(6).strip(),
                        "entrada": match.group(7), "precio_base": parse_num(match.group(8)),
                        "precio_final": parse_num(match.group(9)), "observaciones": (match.group(10) or "").strip(),
                    }
                elif current:
                    current["observaciones"] = (current["observaciones"] + " " + line).strip()
    if current:
        records.append(current)
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df


def extract_resumen(pdf_path: Path) -> pd.DataFrame:
    records, current_section, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if not line:
                    continue
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                    continue
                if not in_summary or line.startswith(RESUMEN_SKIP):
                    continue
                sec = SECTION_PATTERN.match(line)
                if sec:
                    current_section = f"{sec.group(1)} {sec.group(2).strip()}"
                    continue
                match = SUMMARY_ROW_PATTERN.match(line)
                if match:
                    records.append({
                        "pagina": page_num, "seccion": current_section,
                        "edad": match.group(1).strip(), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_med": parse_num(match.group(4)),
                        "minimo": parse_num(match.group(5)), "maximo": parse_num(match.group(6)),
                        "promedio": parse_num(match.group(7)), "vr_animal": parse_num(match.group(8)),
                    })
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df


def feria_procesada(feria: int) -> bool:
    p = LOTE_DIR / f"feria_{feria}.parquet"
    if not p.exists() or p.stat().st_size == 0:
        return False
    try:
        return not pd.read_parquet(p).empty
    except Exception:
        return False


def guardar_feria(lotes_df: pd.DataFrame, resumen_df: pd.DataFrame, feria: int):
    lotes_df = lotes_df.copy()
    lotes_df["fecha_subasta"] = pd.to_datetime(lotes_df["fecha_subasta"], errors="coerce")
    lotes_df.to_parquet(LOTE_DIR / f"feria_{feria}.parquet", index=False)
    if not resumen_df.empty:
        resumen_df = resumen_df.copy()
        resumen_df["fecha_subasta"] = pd.to_datetime(resumen_df["fecha_subasta"], errors="coerce")
        resumen_df.to_parquet(RESUMEN_DIR / f"feria_{feria}.parquet", index=False)


def rebuild_combined():
    lotes_files = sorted(LOTE_DIR.glob("feria_*.parquet"))
    resumen_files = sorted(RESUMEN_DIR.glob("feria_*.parquet"))
    lotes = pd.concat([pd.read_parquet(f) for f in lotes_files], ignore_index=True) if lotes_files else pd.DataFrame()
    resumen = pd.concat([pd.read_parquet(f) for f in resumen_files], ignore_index=True) if resumen_files else pd.DataFrame()
    if not lotes.empty:
        lotes.to_parquet(COMBINED_DIR / "lotes.parquet", index=False)
    if not resumen.empty:
        resumen.to_parquet(COMBINED_DIR / "resumen.parquet", index=False)
    return lotes, resumen


def _procesar_feria(feria: int, force: bool = False):
    if not force and feria_procesada(feria):
        return "skip", feria, None
    pdf = download_pdf(feria)
    lotes_df = extract_lotes(pdf)
    resumen_df = extract_resumen(pdf)
    if lotes_df.empty:
        raise ValueError("sin lotes")
    guardar_feria(lotes_df, resumen_df, feria)
    return "ok", feria, (len(lotes_df), len(resumen_df))


def run_single_feria(feria: int, force: bool = False):
    if not force and feria_procesada(feria):
        print(f"[SKIP] {feria} ya procesada")
        return "skip", feria
    status, num, counts = _procesar_feria(feria, force=force)
    rebuild_combined()
    precargar_datos(rebuild=False, verbose=False)
    print(f"[OK] {num} - {counts[0]} lotes, {counts[1]} resumen")
    return status, num


def run_batch(desde: int, hasta: int, force: bool = False, progress_cb=None):
    if desde > hasta:
        raise ValueError("'Desde' debe ser menor o igual que 'Hasta'")
    ok = skip = fail = 0
    total = hasta - desde + 1
    for i, feria in enumerate(range(desde, hasta + 1), start=1):
        if progress_cb:
            progress_cb(int(i / total * 100), feria)
        if not force and feria_procesada(feria):
            print(f"[SKIP] {feria}")
            skip += 1
            continue
        try:
            _, num, counts = _procesar_feria(feria, force=force)
            print(f"[OK] {num} - {counts[0]} lotes, {counts[1]} resumen")
            ok += 1
        except Exception as e:
            print(f"[FAIL] {feria}: {e}")
            fail += 1
    if progress_cb:
        progress_cb(100, None)
    lotes, resumen = rebuild_combined()
    precargar_datos(rebuild=False, verbose=False)
    print(f"\nResultado: OK={ok} SKIP={skip} FAIL={fail}")
    print(f"Combined: {len(lotes)} lotes | {len(resumen)} filas resumen")
    print(f"Guardado en: {PROYECTO}")
    return ok, skip, fail


CACHE = {"lotes": None, "resumen": None, "summary": None, "cargado": False}


def load_combined_lotes():
    if CACHE.get("cargado") and CACHE.get("lotes") is not None:
        return CACHE["lotes"]
    path = COMBINED_DIR / "lotes.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return rebuild_combined()[0]


def _stats_from_lotes(lotes, ferias):
    info = {
        "ferias": len(ferias),
        "ultima": ferias[-1] if ferias else None,
        "primera": ferias[0] if ferias else None,
        "lotes_total": len(lotes),
        "resumen_ferias": len(list(RESUMEN_DIR.glob("feria_*.parquet"))),
        "fecha_min": None,
        "fecha_max": None,
        "ciudades": [],
    }
    if not lotes.empty and "fecha_subasta" in lotes.columns:
        fechas = pd.to_datetime(lotes["fecha_subasta"], errors="coerce").dropna()
        if not fechas.empty:
            info["fecha_min"] = fechas.min().strftime("%Y-%m-%d")
            info["fecha_max"] = fechas.max().strftime("%Y-%m-%d")
    if not lotes.empty and "ciudad" in lotes.columns:
        info["ciudades"] = sorted(lotes["ciudad"].dropna().unique().tolist())
    return info


def precargar_datos(rebuild=False, verbose=True):
    """Lee Parquet desde Drive a memoria para graficos, tablas y export."""
    ferias = listar_ferias_en_drive()
    if not ferias:
        CACHE.update({"lotes": pd.DataFrame(), "resumen": pd.DataFrame(), "summary": None, "cargado": True})
        msg = "No hay ferias en Drive. Usa la pestaña Descarga."
        if verbose:
            print(msg)
        return {"ok": False, "mensaje": msg, "stats": _stats_from_lotes(pd.DataFrame(), [])}
    combined_path = COMBINED_DIR / "lotes.parquet"
    if rebuild or not combined_path.exists():
        if verbose:
            print("Regenerando consolidado desde ferias en Drive...")
        lotes, resumen = rebuild_combined()
    else:
        if verbose:
            print("Leyendo consolidado desde Drive...")
        lotes = pd.read_parquet(combined_path)
        resumen_path = COMBINED_DIR / "resumen.parquet"
        resumen = pd.read_parquet(resumen_path) if resumen_path.exists() else pd.DataFrame()
    CACHE["lotes"] = lotes
    CACHE["resumen"] = resumen
    CACHE["summary"] = build_price_summary(lotes) if not lotes.empty else None
    CACHE["cargado"] = True
    stats = _stats_from_lotes(lotes, ferias)
    msg = f"Precargado: {stats['lotes_total']:,} lotes | {stats['ferias']} ferias | {stats['fecha_min'] or '?'} → {stats['fecha_max'] or '?'}"
    if verbose:
        print(msg)
    return {"ok": True, "mensaje": msg, "stats": stats}


def datos_precargados():
    return CACHE["cargado"] and CACHE["lotes"] is not None and not CACHE["lotes"].empty


def load_feria_lotes(feria: int):
    path = LOTE_DIR / f"feria_{feria}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"No existe feria_{feria}.parquet")
    return pd.read_parquet(path)


def stats_drive():
    ferias = listar_ferias_en_drive()
    lotes = load_combined_lotes()
    return _stats_from_lotes(lotes, ferias)


MESES = {1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
         7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"}


def build_price_summary(lotes_df=None):
    df = lotes_df.copy() if lotes_df is not None else load_combined_lotes()
    if df.empty:
        raise ValueError("No hay datos de lotes")
    df = df.copy()
    df["fecha_subasta"] = pd.to_datetime(df["fecha_subasta"], errors="coerce")
    df["anio"] = df["fecha_subasta"].dt.year
    df["mes"] = df["fecha_subasta"].dt.month
    df["mes_nombre"] = df["mes"].map(MESES)
    summary = (
        df.groupby(["sexo", "anio", "mes", "mes_nombre"], as_index=False)
        .agg(precio_promedio=("precio_final", "mean"), lotes=("lote", "count"))
        .sort_values(["sexo", "anio", "mes"])
    )
    summary["periodo"] = summary["anio"].astype(str) + "-" + summary["mes"].astype(str).str.zfill(2)
    summary["fecha_periodo"] = pd.to_datetime(summary["periodo"] + "-01")
    summary["pct_mes_anterior"] = summary.groupby("sexo")["precio_promedio"].pct_change() * 100
    summary["pct_mismo_mes_anio_anterior"] = summary.groupby(["sexo", "mes"])["precio_promedio"].pct_change() * 100
    summary["direccion"] = summary["pct_mes_anterior"].apply(
        lambda x: "subio" if pd.notna(x) and x > 0 else ("bajo" if pd.notna(x) and x < 0 else "sin dato")
    )
    return summary


def listar_sexos():
    lotes = load_combined_lotes()
    if lotes.empty or "sexo" not in lotes.columns:
        return []
    return sorted(lotes["sexo"].dropna().unique().tolist())


def plot_precio_tiempo(sexo: str):
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    summary = build_price_summary()
    sub = summary[summary["sexo"] == sexo]
    if sub.empty:
        raise ValueError(f"Sin datos para sexo {sexo}")
    colors = ["#2ca02c" if (v or 0) >= 0 else "#d62728" for v in sub["pct_mes_anterior"].fillna(0)]
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Precio {sexo}", f"Variacion % {sexo}"))
    fig.add_trace(go.Scatter(x=sub["fecha_periodo"], y=sub["precio_promedio"], mode="lines+markers", name="Precio"), row=1, col=1)
    fig.add_trace(go.Bar(x=sub["periodo"], y=sub["pct_mes_anterior"], marker_color=colors, name="% cambio"), row=1, col=2)
    fig.add_hline(y=0, line_dash="dash", row=1, col=2)
    fig.update_layout(height=450, template="plotly_white", title=f"Precio por sexo: {sexo}")
    return fig


def plot_mismo_mes_anios(sexo: str):
    import plotly.graph_objects as go
    summary = build_price_summary()
    sub = summary[summary["sexo"] == sexo]
    if sub.empty:
        raise ValueError(f"Sin datos para sexo {sexo}")
    fig = go.Figure()
    for year in sorted(sub["anio"].unique()):
        ysub = sub[sub["anio"] == year]
        fig.add_trace(go.Bar(x=ysub["mes_nombre"], y=ysub["precio_promedio"], name=str(int(year))))
    fig.update_layout(barmode="group", title=f"Mismo mes entre anos - {sexo}", template="plotly_white", height=450)
    return fig


def show_plotly(fig):
    """Renderiza Plotly en Colab via HTML (funciona fuera de widgets.Output)."""
    from IPython.display import HTML, display
    display(HTML(fig.to_html(include_plotlyjs="cdn")))


def export_csv(include_lotes=True, include_resumen=True):
    if not include_lotes and not include_resumen:
        raise ValueError("Selecciona al menos lotes o resumen")
    if include_lotes:
        lotes_path = COMBINED_DIR / "lotes.parquet"
        if not lotes_path.exists():
            raise FileNotFoundError("No hay combined/lotes.parquet. Descarga ferias primero.")
        lotes = pd.read_parquet(lotes_path)
        lotes.to_csv(EXPORT_DIR / "lotes_consolidado.csv", index=False, encoding="utf-8-sig")
        print(f"CSV lotes: {EXPORT_DIR / 'lotes_consolidado.csv'} ({len(lotes)} filas)")
    if include_resumen:
        resumen_path = COMBINED_DIR / "resumen.parquet"
        if not resumen_path.exists():
            print("Aviso: no hay combined/resumen.parquet")
        else:
            resumen = pd.read_parquet(resumen_path)
            resumen.to_csv(EXPORT_DIR / "resumen_consolidado.csv", index=False, encoding="utf-8-sig")
            print(f"CSV resumen: {EXPORT_DIR / 'resumen_consolidado.csv'} ({len(resumen)} filas)")


ultima = ultima_feria_en_drive()
total = len(listar_ferias_en_drive())
print(f"Listo. Ferias en Drive: {total} | Ultima: {ultima}")
print("Ahora ejecuta la celda de interfaz.")

In [ ]:
# Celda B - Panel interactivo (Descarga · Datos · Exportar)
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

_OUT_STYLE = {"border": "1px solid #ccc", "padding": "8px", "max_height": "360px", "overflow": "auto"}


def _ferias_options():
    f = listar_ferias_en_drive()
    return f if f else [0]


def _stats_html(s):
    return (
        f"<b>Ferias:</b> {s['ferias']} ({s['primera']} → {s['ultima']}) &nbsp;|&nbsp; "
        f"<b>Lotes:</b> {s['lotes_total']:,} &nbsp;|&nbsp; "
        f"<b>Fechas:</b> {s['fecha_min'] or '—'} → {s['fecha_max'] or '—'}<br>"
        f"<b>Ciudades:</b> {', '.join(s['ciudades'][:8])}{'…' if len(s['ciudades']) > 8 else ''}"
    )


_ultima = ultima_feria_en_drive()
_total = len(listar_ferias_en_drive())

ui_estado = widgets.HTML(value="<span style='color:#888'>⏳ Precargando datos...</span>")
btn_precargar = widgets.Button(description="Precargar / refrescar datos", button_style="warning", icon="folder-open")
ui_log_precarga = widgets.Output(layout={"max_height": "72px", "overflow": "auto", "font-size": "12px"})

ui_titulo = widgets.HTML(
    value=(
        "<h3 style='margin-bottom:4px'>Subastas Casanare</h3>"
        f"<p style='color:#555'>Drive: <code>subasta_ganadera_data</code> &nbsp;|&nbsp; "
        f"Ferias: <b>{_total}</b> &nbsp;|&nbsp; Ultima: <b>{_ultima or '—'}</b></p>"
        "<p><small>Graficos → ejecuta <b>Celda C</b></small></p>"
    )
)

ui_desde = widgets.IntText(value=_ultima + 1 if _ultima else 1950, description="Desde:", style={"description_width": "55px"}, layout={"width": "130px"})
ui_hasta = widgets.IntText(value=(_ultima + 50) if _ultima else 2000, description="Hasta:", style={"description_width": "55px"}, layout={"width": "130px"})
ui_force = widgets.Checkbox(value=False, description="Reprocesar existentes")
ui_feria_una = widgets.IntText(value=_ultima + 1 if _ultima else 1950, description="Feria:", style={"description_width": "55px"}, layout={"width": "130px"})
ui_progress = widgets.IntProgress(value=0, min=0, max=100, description="Progreso:", bar_style="info", layout={"width": "98%"})
ui_log_desc = widgets.Output(layout=_OUT_STYLE)

btn_detectar = widgets.Button(description="Detectar ultima", button_style="info", icon="search")
btn_descargar = widgets.Button(description="Descargar rango", button_style="success", icon="download")
btn_una = widgets.Button(description="Descargar 1 feria", icon="cloud-download")
btn_rebuild = widgets.Button(description="Regenerar consolidado", icon="refresh")

ui_feria_ver = widgets.Dropdown(options=_ferias_options(), description="Feria:", style={"description_width": "55px"})
ui_stats = widgets.HTML(value="<i>Los datos se cargan al iniciar la celda.</i>")
ui_log_datos = widgets.Output(layout=_OUT_STYLE)

btn_ver_lotes = widgets.Button(description="Ver lotes", icon="table")
btn_stats = widgets.Button(description="Resumen general", button_style="info", icon="info")
btn_refresh_ferias = widgets.Button(description="Actualizar lista", icon="refresh")

ui_exp_lotes = widgets.Checkbox(value=True, description="CSV lotes")
ui_exp_resumen = widgets.Checkbox(value=True, description="CSV resumen")
ui_log_exp = widgets.Output(layout=_OUT_STYLE)

btn_csv_drive = widgets.Button(description="Guardar CSV en Drive", button_style="success", icon="save")
btn_dl_lotes = widgets.Button(description="Descargar lotes al PC", icon="download")
btn_dl_resumen = widgets.Button(description="Descargar resumen al PC", icon="download")


def _aplicar_precarga(result):
    if result["ok"]:
        ui_estado.value = f"<span style='color:#2e7d32'>✓ {result['mensaje']}</span>"
        ui_stats.value = _stats_html(result["stats"])
    else:
        ui_estado.value = f"<span style='color:#e65100'>⚠ {result['mensaje']}</span>"
        ui_stats.value = "<i>Sin datos. Descarga ferias en la pestaña Descarga.</i>"
    _actualizar_estado()


def on_precargar(_=None):
    btn_precargar.disabled = True
    ui_estado.value = "<span style='color:#888'>⏳ Leyendo Parquet desde Drive...</span>"
    with ui_log_precarga:
        clear_output(wait=True)
        try:
            result = precargar_datos(rebuild=False, verbose=True)
            _aplicar_precarga(result)
        except Exception as e:
            ui_estado.value = f"<span style='color:red'>Error: {e}</span>"
            print(f"Error: {e}")
    btn_precargar.disabled = False


def _actualizar_estado():
    u = ultima_feria_en_drive()
    t = len(listar_ferias_en_drive())
    ui_titulo.value = (
        "<h3 style='margin-bottom:4px'>Subastas Casanare</h3>"
        f"<p style='color:#555'>Drive: <code>subasta_ganadera_data</code> &nbsp;|&nbsp; "
        f"Ferias: <b>{t}</b> &nbsp;|&nbsp; Ultima: <b>{u or '—'}</b></p>"
        "<p><small>Graficos → ejecuta <b>Celda C</b></small></p>"
    )
    opts = _ferias_options()
    ui_feria_ver.options = opts
    if opts and opts != [0]:
        ui_feria_ver.value = opts[-1]


def _set_busy(busy: bool):
    for b in [btn_detectar, btn_descargar, btn_una, btn_rebuild, btn_csv_drive, btn_dl_lotes, btn_dl_resumen]:
        b.disabled = busy


def on_detectar(_):
    u = ultima_feria_en_drive()
    ui_desde.value = u + 1 if u else 1950
    ui_hasta.value = (u + 50) if u else 2000
    ui_feria_una.value = u + 1 if u else 1950
    _actualizar_estado()


def on_descargar(_):
    _set_busy(True)
    ui_progress.value = 0
    with ui_log_desc:
        clear_output(wait=True)
        print(f"Descargando {ui_desde.value} → {ui_hasta.value} ...\n")

        def prog(pct, feria):
            ui_progress.value = pct
            if feria:
                ui_progress.description = f"Feria {feria}:"

        try:
            run_batch(ui_desde.value, ui_hasta.value, force=ui_force.value, progress_cb=prog)
            on_precargar()
        except Exception as e:
            print(f"Error: {e}")
    ui_progress.description = "Progreso:"
    _set_busy(False)


def on_una(_):
    _set_busy(True)
    with ui_log_desc:
        clear_output(wait=True)
        try:
            run_single_feria(ui_feria_una.value, force=ui_force.value)
            on_precargar()
        except Exception as e:
            print(f"Error: {e}")
    _set_busy(False)


def on_rebuild(_):
    with ui_log_desc:
        clear_output(wait=True)
        try:
            lotes, resumen = rebuild_combined()
            print(f"Consolidado: {len(lotes)} lotes | {len(resumen)} resumen")
            on_precargar()
        except Exception as e:
            print(f"Error: {e}")


def on_ver_lotes(_):
    with ui_log_datos:
        clear_output(wait=True)
        try:
            df = load_feria_lotes(ui_feria_ver.value)
            print(f"Feria {ui_feria_ver.value}: {len(df)} lotes\n")
            display(df.head(20))
        except Exception as e:
            print(f"Error: {e}")


def on_stats(_):
    try:
        if not datos_precargados():
            on_precargar()
            return
        s = stats_drive()
        ui_stats.value = _stats_html(s)
        _actualizar_estado()
    except Exception as e:
        ui_stats.value = f"<span style='color:red'>Error: {e}</span>"


def on_refresh_ferias(_):
    on_precargar()


def on_csv_drive(_):
    _set_busy(True)
    with ui_log_exp:
        clear_output(wait=True)
        try:
            export_csv(include_lotes=ui_exp_lotes.value, include_resumen=ui_exp_resumen.value)
        except Exception as e:
            print(f"Error: {e}")
    _set_busy(False)


def on_dl_lotes(_):
    path = EXPORT_DIR / "lotes_consolidado.csv"
    if not path.exists():
        with ui_log_exp:
            clear_output(wait=True)
            print("Primero exporta CSV a Drive (o no existe el archivo).")
        return
    files.download(str(path))


def on_dl_resumen(_):
    path = EXPORT_DIR / "resumen_consolidado.csv"
    if not path.exists():
        with ui_log_exp:
            clear_output(wait=True)
            print("Primero exporta CSV resumen a Drive.")
        return
    files.download(str(path))


btn_precargar.on_click(on_precargar)
btn_detectar.on_click(on_detectar)
btn_descargar.on_click(on_descargar)
btn_una.on_click(on_una)
btn_rebuild.on_click(on_rebuild)
btn_ver_lotes.on_click(on_ver_lotes)
btn_stats.on_click(on_stats)
btn_refresh_ferias.on_click(on_refresh_ferias)
btn_csv_drive.on_click(on_csv_drive)
btn_dl_lotes.on_click(on_dl_lotes)
btn_dl_resumen.on_click(on_dl_resumen)

tab_descarga = widgets.VBox([
    widgets.HTML("<b>Descarga masiva</b>"),
    widgets.HBox([ui_desde, ui_hasta, ui_force]),
    widgets.HBox([btn_detectar, btn_descargar]),
    widgets.HTML("<b>Una feria</b>"),
    widgets.HBox([ui_feria_una, btn_una, btn_rebuild]),
    ui_progress,
    widgets.HTML("<b>Log:</b>"),
    ui_log_desc,
])

tab_datos = widgets.VBox([
    widgets.HBox([ui_feria_ver, btn_ver_lotes, btn_refresh_ferias]),
    widgets.HBox([btn_stats]),
    ui_stats,
    widgets.HTML("<b>Vista previa:</b>"),
    ui_log_datos,
])

tab_exportar = widgets.VBox([
    widgets.HBox([ui_exp_lotes, ui_exp_resumen]),
    widgets.HBox([btn_csv_drive, btn_dl_lotes, btn_dl_resumen]),
    widgets.HTML("<b>Log:</b>"),
    ui_log_exp,
])

tabs = widgets.Tab(children=[tab_descarga, tab_datos, tab_exportar])
tabs.set_title(0, "Descarga")
tabs.set_title(1, "Datos")
tabs.set_title(2, "Exportar")

display(widgets.VBox([
    ui_titulo,
    widgets.HBox([btn_precargar, ui_estado]),
    ui_log_precarga,
    tabs,
]))

on_precargar()


## Celda C — Graficos

Ejecuta **despues de Celda A y B**. Usa los menus; el grafico aparece en esta celda.


In [ ]:
# Celda C - Graficos (Output dedicado — funciona en Colab)
from IPython.display import HTML, display, clear_output
from ipywidgets import Dropdown, HBox, VBox, Output, interactive_output

if not datos_precargados():
    precargar_datos(rebuild=False, verbose=True)

sexos = listar_sexos()
if not sexos:
    print("No hay datos. Usa Celda B → Descarga ferias, luego vuelve aqui.")
else:
    ui_vista = Dropdown(
        options=["Precio en el tiempo", "Mismo mes entre anos", "Tabla analitica"],
        description="Vista:",
        style={"description_width": "45px"},
        layout={"width": "280px"},
    )
    ui_sexo = Dropdown(
        options=sexos,
        description="Sexo:",
        value=sexos[0],
        style={"description_width": "45px"},
        layout={"width": "180px"},
    )
    graf_out = Output(layout={"border": "1px solid #ddd", "padding": "8px", "width": "100%"})

    def _mostrar_grafico(vista, sexo):
        with graf_out:
            clear_output(wait=True)
            try:
                if vista == "Tabla analitica":
                    summary = CACHE.get("summary")
                    if summary is None:
                        summary = build_price_summary()
                    sub = summary[summary["sexo"] == sexo]
                    cols = [
                        "periodo", "precio_promedio", "pct_mes_anterior",
                        "direccion", "pct_mismo_mes_anio_anterior",
                    ]
                    display(sub[cols].round(1).tail(24))
                else:
                    fig = (
                        plot_precio_tiempo(sexo)
                        if vista == "Precio en el tiempo"
                        else plot_mismo_mes_anios(sexo)
                    )
                    display(HTML(fig.to_html(include_plotlyjs="cdn")))
            except Exception as e:
                print(f"Error: {e}")

    interactive_output(_mostrar_grafico, {"vista": ui_vista, "sexo": ui_sexo})
    display(VBox([HBox([ui_vista, ui_sexo]), graf_out]))
    _mostrar_grafico(ui_vista.value, ui_sexo.value)
